# MolQL selectors

MolViewSpec accepts eagerly built base MolQL expression trees for component and color selectors and for primitive positions. PyMOL text is transpiled when the state is created, so the MVSJ contains only JSON MolQL.

In [ ]:
# !pip install molviewspec
# !pip install -e ../../molviewspec

In [4]:
from molviewspec import create_builder, molql, molstar_notebook, MVSJ

## Build and transpile queries

Named MolQL arguments retain their canonical hyphenated spelling. The PyMOL query is parsed immediately to the same base-language JSON representation.

In [19]:
imatinib = molql.struct.generator.atom_groups({
    "chain-test": molql.core.rel.eq([
        molql.struct.atom_property.macromolecular.label_asym_id(),
        "G",
    ]),
})
imatinib_n13 = molql.struct.generator.atom_groups({
    "chain-test": molql.core.rel.eq([
        molql.struct.atom_property.macromolecular.label_asym_id(),
        "G",
    ]),
    "atom-test": molql.core.rel.eq([
        molql.struct.atom_property.macromolecular.label_atom_id(),
        "N13",
    ]),
})
binding_pocket = molql.from_pymol("byres polymer within 5 of resn STI")
thr315_og1 = molql.from_pymol("chain A and resi 315 and name OG1")
thr315 = molql.from_pymol("chain A and resi 315")

## Use MolQL in an MVS state

In [25]:
builder = create_builder()
structure = (
    builder.download(url="https://files.wwpdb.org/download/1iep.cif")
    .parse(format="mmcif")
    .assembly_structure(ref="structure")
)

polymer = structure.component(selector="polymer").representation(type="cartoon")
polymer.color(color="#8AA6C1")
polymer.color(color="#B8497A", selector=molql.selector(binding_pocket))
polymer.color(color="red", selector=molql.selector(thr315))

structure.component(selector=molql.selector(imatinib)).representation(
    type="ball_and_stick"
).color(color="#F08A4B")

structure.component(selector=molql.selector(thr315)).representation(
    type="ball_and_stick"
).color(color="red")

structure.primitives().distance(
    start=molql.position(imatinib_n13),
    end=molql.position(thr315_og1),
    color="#F08A4B",
    dash_length=0.2,
    label_template="Imatinib N13–Thr315 OG1: {{distance}}",
)

state = builder.get_state(title="MolQL selectors")

# MVSJ(data=state).dump("tmp/08_molql.mvsj")
# molstar_notebook(state)